# Multi-Agent Coordination and Workflow Control — Starter

This combined starter notebook covers both parts of Module 5:

- **Part 1** – Multi-Agent Coordination and Communication.
  You will design a three-agent LangGraph pipeline and experiment with one-way, two-way, and brainstorming protocols.

- **Part 2** – Refining Coordination, Workflow Control, and Benchmarking.
  You will extend the workflow with MCP-style tools, conditional routing, iteration loops, failover handling, and simple benchmarks for different coordination strategies.

You can work through the notebook top-to-bottom, or focus on one part at a time.

## Part 1: Multi-Agent Coordination and Communication — Starter

In this guided lab, you will design a simple multi-agent workflow using LangGraph.
You will decide how many agents you need, define their roles, and compare communication protocols.

## What is Agent Coordination?

Agent coordination refers to how multiple AI agents work together to solve a problem. Key design questions include:

- **How many agents do we need?** (Fewer = simpler, more = richer perspectives but higher overhead)
- **What does each agent own?** (Shared state vs. private responsibilities)
- **How do agents communicate?** (One-way broadcast, bidirectional Q&A, or brainstorming?)
- **What is the execution order?** (Sequential: A→B→C, Parallel, or conditional routing?)

This lab explores these dimensions empirically by building, running, and comparing different coordination strategies.

## Real-World Coordination Topologies

Research and production systems have converged on a few standard multi-agent topologies.
This lab implements the simplest — a **linear pipeline** — then the second part of this module extends it toward the others:

| Topology | Structure | When to use |
|----------|-----------|-------------|
| **Linear pipeline** | A → B → C (this lab) | Well-understood tasks with clear sequential dependencies |
| **Supervisor** | Central orchestrator routes to specialist agents | Tasks requiring dynamic routing; orchestrator decides which agent runs next |
| **Swarm / peer-to-peer** | Agents hand off directly to each other | Decentralized workflows; no single bottleneck; higher token cost |
| **Ensemble / majority vote** | Same task sent to N independent agents; outputs aggregated | Reliability-critical tasks; research shows this consistently improves accuracy on reasoning problems |

> **Research note:** LangGraph natively supports all four topologies. The Supervisor and Swarm patterns
> are documented in the official LangGraph multi-agent concepts guide and each has a prebuilt helper
> (`create_react_agent`, `create_swarm`). Empirical work (Li et al., 2024, "More Agents Is All You Need")
> found that even simple majority-vote ensembles reliably improve performance on math and coding benchmarks —
> a result that suggests coordination design is at least as important as individual model quality.

## Lab Goals

By the end of this lab, you will:

1. Determine the minimum number of agents needed for a task
2. Define clear role responsibilities and information ownership
3. Build a LangGraph workflow with explicit execution dependencies
4. Implement and compare three communication protocols:
   - **One-way:** Sequential, no feedback
   - **Two-way:** Agents ask clarifying questions
   - **Brainstorm:** Agents generate multiple alternatives
5. Measure how protocol choice affects output quality and coordination cost

---
## Lab Logistics

| | |
|---|---|
| **Estimated time** | 45–60 minutes |
| **Format** | Live guided — instructor leads, you implement |
| **Prerequisites** | Python basics (functions, dictionaries, loops) |
| **API key required?** | No — baseline uses simulated agents (no LLM calls needed) |

### How to work through this notebook
1. **Read** every markdown cell fully before running any code
2. **Find** `📝 TODO` markers — those are your hands-on implementation tasks
3. **Run cells in order** top-to-bottom (use *Kernel → Restart & Run All* to reset)
4. **Pause at each Part boundary** — your instructor will facilitate a brief group discussion

> ⏱️ **Instructor pacing guide:**
> - Part A — Agent design decisions: ~10 min
> - Part B — Implement agent functions: ~15 min
> - Part C — Build graph + run all protocols: ~15 min
> - Analysis + Reflection: ~15 min

### Key concepts in this lab
| Concept | What it means |
|---------|--------------|
| **Agent** | A function that reads shared state and writes back outputs |
| **State** | A shared dictionary all agents read from and write to |
| **Graph** | Explicit wiring that defines which agent runs after which |
| **Protocol** | The communication pattern governing how much back-and-forth occurs |

In [ ]:
!pip install -q langgraph

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


class WorkflowState(TypedDict):
    task: str
    protocol: Literal['one_way', 'two_way', 'brainstorm']
    planner_notes: str
    researcher_notes: str
    writer_output: str
    message_log: list[str]


print('✓ Imports and state ready')

## Part A: Determine Agent Count and Roles

### The Agent Design Trade-off

**Adding agents:** More perspectives ✓ but more coordination overhead ✗

**Task:** Create a 5-step launch plan for a new AI tutoring product for college students.

When sizing your agent team, consider:
- **What expertise is essential?** (e.g., strategy, market validation, communication)
- **Can any roles be merged?** (e.g., does "researcher" and "validator" need to be separate?)
- **What is the minimal set that covers all viewpoints without redundancy?**
- **Will adding one more agent significantly improve quality?** (Diminishing returns curve)

**📝 TODO:** Decide on 2–4 agents and define each role's responsibility in 1–2 sentences. Modify the baseline roles below if you think they need adjustment.
Consider: would merging two roles lose important perspective? Would splitting gain clarity?

In [ ]:
task = 'Create a 5-step launch plan for a new AI tutoring product for college students.'

# TODO: Customize these roles or add/remove agents as you see fit
# Suggested format: [{'name': 'role_name', 'responsibility': '...'}, ...]
agent_roles = [
    {'name': 'planner', 'responsibility': 'Define launch phases, sequencing, and dependencies.'},
    {'name': 'researcher', 'responsibility': 'Identify market risks, validate assumptions, add evidence.'},
    {'name': 'writer', 'responsibility': 'Synthesize inputs into a concise, actionable 5-step plan.'},
]

assert len(agent_roles) >= 2, 'Use at least two agents for this lab.'
assert all(role['responsibility'] for role in agent_roles), 'Each role must have a clear responsibility.'

print('Agent count:', len(agent_roles))
print('\nRoles:')
for role in agent_roles:
    print(f'  • {role["name"]}: {role["responsibility"]}')

## Part B: Implement Agent Functions

### Communication Patterns Explained

Each agent reads from shared `WorkflowState` and writes back its outputs. The **protocol** field determines how much back-and-forth happens:

**One-way Protocol:** Planner → Researcher → Writer (minimal overhead, no loops)
- Simple and fast: each agent processes input once
- **Trade-off:** Early decisions can't be questioned; later agents inherit any upstream errors
- **Best for:** Time-critical decisions, high-trust teams

**Two-way Protocol:** Agents ask clarifying questions and iterate
- Enables alignment: "Researcher asks Planner for clarification → Planner responds"
- **Trade-off:** Adds latency and complexity but catches misalignments
- **Best for:** High-stakes decisions, uncertain assumptions, disagreements expected

**Brainstorm Protocol:** Agents generate multiple alternative ideas
- Explores solution space: Researcher offers "3 alternative go-to-market angles"
- **Trade-off:** Much higher coordination cost (more messages, more ideas to evaluate)
- **Best for:** Creative phases, exploring unknowns, innovation

**📝 TODO:** Fill in each function with protocol-aware logic. Use `state['protocol']` to branch behavior. The baseline shows patterns; extend them with your own details tailored to `state['task']`.

In [ ]:
def planner_agent(state: WorkflowState) -> WorkflowState:
    # TODO: Replace with your own planning logic—be specific to state['task']
    planner_notes = (
        '1) Define target student segments and value prop; '
        '2) Run pilot with 2 departments with clear KPIs; '
        '3) Prepare ambassador and referral channels for scale.'
    )
    state['planner_notes'] = planner_notes
    state['message_log'].append(f"planner -> researcher: {planner_notes}")
    return state


def researcher_agent(state: WorkflowState) -> WorkflowState:
    # TODO: Add your own risk analysis and market validation logic
    base_notes = (
        'Risks: early churn, privacy compliance, faculty adoption lag. '
        'Evidence needed: pilot completion rate, week-4 retention, NPS by major.'
    )

    if state['protocol'] == 'one_way':
        # One-way: just consume planner notes, add findings
        researcher_notes = base_notes
    elif state['protocol'] == 'two_way':
        # Two-way: ask planner a clarifying question, simulate their response
        researcher_notes = base_notes + ' Q: Prioritize growth or retention in phase 1?'
        state['message_log'].append('researcher → planner: Growth or retention priority?')
        state['message_log'].append('planner → researcher: Retention first to validate product fit.')
    else:  # brainstorm
        # Brainstorm: generate multiple alternative angles
        researcher_notes = (
            base_notes + ' Alternatives: A) Faculty-led onboarding, '
            'B) Peer study-group integrations, C) Exam-season campaigns.'
        )
        state['message_log'].append('researcher brainstorm: 3 alternative pillars generated')

    state['researcher_notes'] = researcher_notes
    state['message_log'].append(f"researcher → writer: [findings ready]")
    return state


def writer_agent(state: WorkflowState) -> WorkflowState:
    # TODO: Synthesize planner + researcher into final 5-step plan
    writer_output = f"""=== Final 5-Step Launch Plan ===
1. Segment: Prioritize student cohorts and departments based on demand.
2. Pilot: Run 2-department pilot with clear success metrics and KPIs.
3. Validate: Track retention, NPS, and completion against risk assumptions.
4. Expand: Scale through ambassadors and referrals where pilot wins strongest.
5. Operationalize: Build faculty onboarding, privacy/compliance, and support.

Planner strategy: {state['planner_notes']}
Research insights: {state['researcher_notes']}
Protocol used: {state['protocol']}
"""
    state['writer_output'] = writer_output
    state['message_log'].append('writer: final output ready')
    return state

print('✓ Agent functions defined')

## Part C: Build Coordination Flow with LangGraph

### Explicit Dependencies in LangGraph

LangGraph lets you define a **computation graph** where:
- **Nodes** = agent functions or other logic
- **Edges** = execution order and data flow

This makes the workflow **explicit and auditable**:

```
  START
    ↓
  planner (generates strategy and phases)
    ↓
  researcher (adds validation and risks)
    ↓
  writer (synthesizes into final plan)
    ↓
   END
```

**Benefits:**
- Clear data flow (no hidden dependencies)
- Easy to visualize and debug
- Scalable to complex conditional logic (covered in Part 2 of this module)

**📝 TODO:** Build the graph with `add_edge()` calls to connect START → planner → researcher → writer → END. Then invoke the compiled graph with all three protocols and observe differences.

In [ ]:
def build_graph():
    graph = StateGraph(WorkflowState)
    
    # TODO: Add nodes for each agent
    graph.add_node('planner', planner_agent)
    graph.add_node('researcher', researcher_agent)
    graph.add_node('writer', writer_agent)
    
    # TODO: Add edges to define execution order
    graph.add_edge(START, 'planner')
    graph.add_edge('planner', 'researcher')
    graph.add_edge('researcher', 'writer')
    graph.add_edge('writer', END)
    
    return graph.compile()


compiled_graph = build_graph()
print('✓ Graph compiled successfully')

In [ ]:
# Visualize the compiled workflow graph
# LangGraph can render a Mermaid diagram when graphviz is available
try:
    from IPython.display import Image, display
    img = compiled_graph.get_graph().draw_mermaid_png()
    display(Image(img))
    print("✓ Graph rendered above")
except Exception:
    # Fallback: ASCII art diagram when rendering is unavailable
    print("""
    Workflow Execution Order (data flows top-to-bottom)
    ─────────────────────────────────────────────────────

           ┌──────────────┐
           │    START     │
           └──────┬───────┘
                  │
                  ▼
         ┌────────────────┐
         │    planner     │  ← defines strategy and launch phases
         └────────┬───────┘
                  │  planner_notes written to shared state
                  ▼
         ┌────────────────┐
         │   researcher   │  ← validates assumptions; protocol-dependent back-and-forth
         └────────┬───────┘
                  │  researcher_notes written to shared state
                  ▼
         ┌────────────────┐
         │     writer     │  ← synthesizes final 5-step launch plan
         └────────┬───────┘
                  │
                  ▼
           ┌──────────────┐
           │     END      │
           └──────────────┘

    Shared WorkflowState fields:
      task            → the problem all agents are solving
      protocol        → one_way | two_way | brainstorm
      planner_notes   → written by planner, read by researcher + writer
      researcher_notes→ written by researcher, read by writer
      writer_output   → final synthesized plan
      message_log     → audit trail of all inter-agent messages
    """)

In [ ]:
def run_protocol(protocol: str):
    initial_state: WorkflowState = {
        'task': task,
        'protocol': protocol,
        'planner_notes': '',
        'researcher_notes': '',
        'writer_output': '',
        'message_log': []
    }
    return compiled_graph.invoke(initial_state)


results = {}
for protocol in ['one_way', 'two_way', 'brainstorm']:
    results[protocol] = run_protocol(protocol)
    print(f"\n{'='*60}")
    print(f"PROTOCOL: {protocol.upper()}")
    print(f"{'='*60}")
    print('\nFinal Output:')
    print(results[protocol]['writer_output'])
    print('\nMessages Exchanged:')
    for msg in results[protocol]['message_log']:
        print(f"  • {msg}")
    print(f"\nTotal messages: {len(results[protocol]['message_log'])}")

In [ ]:
import matplotlib.pyplot as plt

# ─── Protocol Comparison: Visual Summary ───────────────────────────────────────
protocols = ['one_way', 'two_way', 'brainstorm']
colors = ['#4C9BE8', '#F5A623', '#7ED321']

message_counts = [len(results[p]['message_log']) for p in protocols]

# Coordination richness proxy: number of explicit back-and-forth exchanges
back_and_forth = {'one_way': 0, 'two_way': 1, 'brainstorm': 2}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Module 5 — Part 1 Protocol Comparison', fontsize=13, fontweight='bold', y=1.02)

# Left chart: total messages exchanged
ax1 = axes[0]
bars1 = ax1.bar(protocols, message_counts, color=colors, width=0.5)
ax1.set_title('Total Messages Exchanged', fontweight='bold')
ax1.set_ylabel('Message count')
ax1.set_ylim(0, max(message_counts) + 2)
for bar, count in zip(bars1, message_counts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             str(count), ha='center', va='bottom', fontweight='bold')

# Right chart: coordination feedback rounds
ax2 = axes[1]
btf_vals = [back_and_forth[p] for p in protocols]
bars2 = ax2.bar(protocols, btf_vals, color=colors, width=0.5)
ax2.set_title('Coordination Loops (Feedback Round-Trips)', fontweight='bold')
ax2.set_ylabel('Round trips')
ax2.set_ylim(0, max(btf_vals) + 1)
for bar, val in zip(bars2, btf_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Text comparison table
descriptions = {
    'one_way':    'Speed, low cost',
    'two_way':    'Alignment, quality',
    'brainstorm': 'Exploration, creativity',
}
print(f"\n{'─'*65}")
print(f"{'Protocol':<14} {'Messages':>10} {'Feedback loops':>16} {'Best for':>20}")
print(f"{'─'*65}")
for p in protocols:
    print(f"{p:<14} {len(results[p]['message_log']):>10} {back_and_forth[p]:>16} {descriptions[p]:>20}")
print(f"{'─'*65}")
print("\n🔍 Pause & Discuss: Which protocol produced the most useful plan for this task?")
print("   What would you choose if you had a 5-minute deadline vs. a 5-day deadline?")

### Try Your Own Task (Part 1)

Use this notebook as a sandbox for your own coordination problems:

1. Change the `task` description near the top of Part 1 to something from your own work.
2. Edit `agent_roles` so the responsibilities match the real stakeholders you would involve.
3. Re-run the Part 1 cells (imports → state → agents → graph → protocol runs → visualization).
4. For each protocol, jot down:
   - What changed in the final plan?
   - How many messages were exchanged?
   - Which protocol would you choose in practice, and why?

### Quick Start Demo (Part 1)

After running the setup cells above (imports, `WorkflowState`, agent implementations, and `build_graph`),
you can use this cell to quickly run the Part 1 workflow for a single protocol.

- Start with the default task, then optionally edit it to something closer to your domain.
- Change the protocol name to see how coordination changes.

In [ ]:
# Quick start: run Part 1 once for a single protocol
example_protocol = 'two_way'  # change to 'one_way' or 'brainstorm' if you like
quick_state = compiled_graph.invoke({
    'task': task,
    'protocol': example_protocol,
    'planner_notes': '',
    'researcher_notes': '',
    'writer_output': '',
    'message_log': []
,})

print(f"Protocol: {example_protocol}\n")
print("Final output (truncated to first 800 chars):\n")
print(quick_state['writer_output'][:800])
print("\nMessages exchanged:")
for msg in quick_state['message_log']:
    print("  •", msg)
print(f"\nTotal messages: {len(quick_state['message_log'])}")

## What Research Says About Coordination Protocols

Your simulation above models patterns that researchers have studied empirically in real LLM-agent systems.
Here is how the literature maps to what you just observed:

### "More Agents Is All You Need" — Li et al. (2024)

This paper ([arXiv 2402.05120](https://arxiv.org/abs/2402.05120)) introduced **Agent Forest**: spawn many independent
agent instances with the same task, collect all responses, and pick the best by majority vote.

**Key finding:** simply scaling the number of agents consistently improves performance — without any complex
coordination logic. On the GSM8K math benchmark:

| Model | Agents | Accuracy |
|-------|--------|----------|
| Llama2-13B | Many (Agent Forest) | **59%** |
| Llama2-70B | 1 (baseline) | 54% |

A *smaller* model with more agents **outperformed** a much larger model running alone.
This is the empirical case for why coordination design matters as much as model quality.

**Practical implication for this lab:** your `brainstorm` protocol, which generates multiple alternative
angles and relies on the writer to synthesize, is the closest simulation of this ensemble effect.

### LangChain Multi-Agent Architecture Benchmark (2024)

LangChain tested three architectures — Single Agent, Supervisor, and Swarm — on a modified version of
the τ-bench dataset with synthetic distractor domains added to stress-test scalability.
([Source: blog.langchain.com](https://blog.langchain.com/benchmarking-multi-agent-architectures/))

**Results:**

| Architecture | Performance with distractors | Token efficiency |
|--------------|------------------------------|-----------------|
| Single Agent | Degrades sharply at 2+ distractor domains | Grows with distractors |
| Supervisor | Moderate; initially underperformed Swarm | Flat — scales well |
| Swarm | Best overall | Flat — most efficient |

The Supervisor architecture improved by **nearly 50%** once three optimizations were applied:
removing unnecessary handoff messages, adding a "forwarding messages" tool, and renaming handoff tools.

> **Takeaway:** architecture topology is not a fixed property — small implementation choices
> (like whether a supervisor regenerates or forwards a sub-agent response) can swing performance by 50%.

### Communication Overhead and Protocol Choice

A 2024 paper "Which LLM Multi-Agent Protocol to Choose?" found that protocol selection alone
can impact **task completion time by up to 36%** and introduces **communication overhead of up to 3.5 seconds**
per agent interaction. Hybrid protocol deployments outperformed homogeneous ones by approximately **6.6%**.
([Source: arXiv 2510.17149](https://arxiv.org/pdf/2510.17149))

This is why the protocol comparison you just ran matters: even in simulation, the right protocol
for a given task type is not obvious and must be benchmarked, not assumed.

## Industry Snapshot: Where Multi-Agent AI Stands Today

### LangChain State of AI Agents Report (2024)

LangChain surveyed **1,300+ professionals** — developers, product managers, and business leaders —
on how they are using AI agents in practice.
([Source: langchain.com/stateofaiagents](https://www.langchain.com/stateofaiagents))

| Metric | Finding |
|--------|---------|
| Agents in production today | **51%** of respondents |
| Planning to implement agents | **78%** of respondents |
| Top use case | Research & summarization (**58%**) |
| Second top use case | Personal productivity / assistance (**53.5%**) |
| Avg. steps per workflow trace (2023) | 2.8 steps |
| Avg. steps per workflow trace (2024) | **7.7 steps** (+175%) |
| Using 2+ control methods | 51% of tech respondents |
| Non-tech companies deploying agents | **90%** have or plan to |

**The 7.7 steps finding is significant:** workflows are becoming substantially more complex year-over-year,
which is exactly why topics like explicit coordination, conditional routing, and benchmarking across both parts of this module
are becoming essential engineering skills rather than research curiosities.

### The Four LangGraph Topology Patterns (as of 2025)

LangChain has published four named patterns for building multi-agent systems with LangGraph, each with
tradeoffs covered across this part and the later Part 2 of this module:

| Pattern | Key idea | LangGraph primitives |
|---------|----------|---------------------|
| **Network / pipeline** | Agents linked in a graph with defined edges (this lab) | `add_edge`, `StateGraph` |
| **Supervisor** | A coordinator LLM decides which sub-agent to invoke next | `add_conditional_edges`, `create_react_agent` |
| **Hierarchical** | Supervisors can themselves delegate to sub-supervisors (trees of coordinators) | Nested `StateGraph` |
| **Swarm** | Agents hand off to each other directly via tool calls, no central node | `create_swarm()`, `create_handoff_tool()` |

> **Where you are in the landscape:** this lab implements the Network / Pipeline pattern.
> The second part of this module adds conditional logic that begins to resemble the Supervisor pattern.
> Full Supervisor and Swarm implementations are natural extensions from here.

## Reflection & Analysis

Use these questions to deepen your understanding of coordination trade-offs:

1. **Output Quality:** Which protocol produced the clearest, most actionable plan?
   - Look for: specificity, completeness, justified decisions, ownership clarity
   - Rate from 1–5 for each protocol

2. **Coordination Cost:** Which protocol introduced the most overhead?
   - Count total messages exchanged
   - Measure latency (how many sequential steps?)
   - Which felt most "natural" to follow?

3. **Trade-off Analysis:** Did communication richness justify added complexity?
   - Quality gain = two_way_quality - one_way_quality
   - Cost increase = two_way_messages - one_way_messages
   - ROI = quality_gain / cost_increase (rough metric)
   - Is it positive? Would you use two-way by default?

4. **Production Decision:** If launching this workflow in a real team, which protocol would you choose?
   - Time pressure: (fast = one-way, thorough = two-way/brainstorm)
   - Team structure: (co-located = two-way OK, distributed = one-way simpler)
   - Risk tolerance: (high-stakes = two-way, low-stakes = one-way)
   - Budget: (expensive agents = minimize calls = one-way)

5. **Scaling:** How would decisions change if you added a 4th agent (e.g., customer researcher)?
   - Would coordination overhead grow 2-way O(n²)?
   - At what team size does two-way become impractical?
   - Would you use hierarchical routing instead?

---
## Summary & Transition to Part 2

**What you built in this lab:**
- A 3-agent LangGraph workflow with explicit execution dependencies
- Three communication protocols with observable, measurable trade-offs
- An empirical comparison of message overhead vs. coordination richness

**Design principles you practiced:**

| Principle | Key question to ask |
|-----------|-------------------|
| Agent count | What is the minimum viable team that covers all essential viewpoints? |
| Role boundaries | Does each agent own a distinct piece of information or expertise? |
| Communication | Does the task require alignment (two-way) or exploration (brainstorm)? |
| Graph explicitness | Can someone reading the graph understand the full data flow? |

**Protocol selection cheatsheet:**

```
Low stakes, fast turnaround   →  one_way
High stakes, uncertain assumptions  →  two_way
Creative phase, unknown solution space  →  brainstorm
```

**Before you move on from Part 1, you should be able to:**
- Describe when you would use each protocol (one-way, two-way, brainstorm) for your own task.
- Sketch the LangGraph for Part 1 and label what each state field represents.
- Explain how message count and "back-and-forth" affect coordination cost.

---

**In Part 2 of this module**, you will extend this workflow with:
- **Conditional routing** — "if tool call fails, take the failover path"
- **Iteration loops** — "keep revising until quality threshold is met"
- **Error handling** — graceful recovery when tools fail
- **Benchmarking** — quantitative comparison of coordination strategies
- **MCP tools** — standardized external tool access pattern

> 💡 **Carry your intuitions forward:** The overhead-vs-quality trade-off you observed here will appear again in the benchmark results from the second part of this module — but now you'll have numbers to back it up.

---
## Further Reading & Resources

### LangGraph Documentation
- [LangGraph Multi-Agent Concepts](https://langchain-ai.github.io/langgraph/concepts/multi_agent/) — Supervisor, Swarm, Handoff, and pipeline patterns with code examples
- [LangGraph Low-Level API](https://langchain-ai.github.io/langgraph/concepts/low_level/) — `StateGraph`, `add_edge`, `add_conditional_edges`, and graph compilation

### Research Papers
- [More Agents Is All You Need](https://arxiv.org/abs/2402.05120) (Li et al., 2024) — Ensemble with sampling and voting; Llama2-13B beats 70B with more agents
- [Which LLM Multi-Agent Protocol to Choose?](https://arxiv.org/pdf/2510.17149) (2024) — Quantified impact of protocol selection on task completion time (+36%) and communication overhead
- [A Communication-Centric Survey of LLM-Based Multi-Agent Systems](https://arxiv.org/pdf/2502.14321) (2025) — Taxonomy of agent communication patterns: cooperative, competitive, hybrid
- [AgentBench](https://arxiv.org/abs/2308.03688) (Liu et al., 2023) — Systematic benchmark across 8 agent task types

### Industry Reports & Blogs
- [LangChain Benchmarking Multi-Agent Architectures](https://blog.langchain.com/benchmarking-multi-agent-architectures/) — Empirical comparison of Single Agent, Supervisor, and Swarm on τ-bench
- [LangChain State of AI Agents Report 2024](https://www.langchain.com/stateofaiagents) — Survey of 1,300+ professionals: 51% using agents in production; workflow complexity +175% YoY
- [Building Multi-Agent Systems with LangGraph Swarm](https://dev.to/sreeni5018/building-multi-agent-systems-with-langgraph-swarm-a-new-approach-to-agent-collaboration-15kj) — Practical guide to the swarm topology

## Part 2: Refining Coordination, Workflow Control, and Benchmarking — Starter

In this lab, you extend the Part 1 workflow with **adaptive control logic**: conditional routing, iteration loops, error handling, and tool standardization.

## From Static Flows to Adaptive Workflows

In Part 1, execution was **deterministic**: planner → researcher → writer, always in that order.

But production workflows must be **adaptive**:
- **Handle errors:** If a tool fails, take a fallback path instead of crashing
- **Iterate when needed:** If output quality is low, revise and re-evaluate
- **Route conditionally:** Skip steps that aren't needed for this specific task
- **Access tools predictably:** Via a standardized interface (MCP) so agents don't hardcode integrations

This lab introduces patterns for control flow that make workflows **resilient and data-driven**.

## Lab Goals

1. Implement **conditional routing** in LangGraph (decisions based on state)
2. Add **iteration and retry logic** (loops until quality threshold)
3. Design **error handling paths** (graceful fallback when tools fail)
4. **Benchmark** different coordination strategies on reliability and efficiency
5. Integrate **MCP-style tool calling** (standardized interface for external tools)

By the end, you'll understand when to use each strategy and how to measure trade-offs in production.

---
## Lab Logistics

| | |
|---|---|
| **Estimated time** | 45–60 minutes |
| **Format** | Live guided — extends Part 1 concepts |
| **Prerequisites** | Part 1 of this module completed (LangGraph basics, agent roles, protocol comparison) |
| **API key required?** | No |

### How to use this notebook
1. **Read** each markdown section before running code
2. **Find** `📝 TODO` markers — these are your implementation tasks
3. **Run cells in order** (use *Kernel → Restart & Run All* to start fresh)
4. **Pause at each Part** for a group discussion with your instructor

> ⏱️ **Instructor pacing guide:**
> - Part A — MCP tool interface: ~10 min
> - Part B — Control nodes: ~15 min
> - Part C — Conditional graph: ~10 min
> - Part D — Benchmark + visualize: ~15 min
> - Reflection: ~10 min

### Part 2 vs Part 1 at a glance

| Feature | Part 1 | Part 2 |
|---------|--------|--------|
| Execution flow | Always linear | Conditional (branches) |
| Error handling | None | Failover path |
| Revision loops | None | Iterative retry with cap |
| Tool access | Hardcoded | MCP-style standard interface |
| Evaluation | Qualitative | Quantitative benchmarks |

In [7]:
!pip install -q langgraph pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
import pandas as pd


class ControlState(TypedDict):
    task: str
    protocol: Literal['one_way', 'two_way', 'brainstorm']
    draft: str
    feedback: str
    final_output: str
    quality_score: float
    retries: int
    max_retries: int
    needs_tool: bool
    error: str
    log: list[str]


print('✓ Imports and state ready')

✓ Imports and state ready


## Part A: Add an MCP-Style Tool Interface

### What is MCP?

The **Model Context Protocol (MCP)** is an open standard introduced by Anthropic in **November 2024**.
It defines a universal interface for connecting AI models and agents to external tools, data sources,
and services — replacing the fragmented ecosystem of one-off integrations with a single standardized contract.

**Analogy:** MCP is to AI tools what **USB-C** is to device connectors — one universal port
for any tool, any agent, any host application.

**The problem it solves (the "M×N problem"):**
Without MCP, integrating M different LLMs with N different tools requires M×N custom connectors.
With MCP, every LLM speaks one protocol and every tool implements one protocol: M + N integrations total.

### MCP Architecture

```
┌─────────────────────────────────────────────────┐
│  MCP Host (your agent / LangGraph workflow)      │
│   ┌─────────────┐    ┌─────────────────────┐    │
│   │ MCP Client  │◄──►│  MCP Client         │    │
│   └──────┬──────┘    └──────────┬──────────┘    │
└──────────┼───────────────────────┼───────────────┘
           │ stdio / HTTP+SSE      │
   ┌───────▼──────┐        ┌───────▼──────┐
   │  MCP Server  │        │  MCP Server  │
   │ (local tool) │        │ (remote API) │
   └──────────────┘        └──────────────┘
```

**MCP's three core primitives:**

| Primitive | Controlled by | What it exposes | Example |
|-----------|--------------|----------------|---------|
| **Tools** | Model | Executable functions the LLM can call | `run_sql_query`, `estimate_budget` |
| **Resources** | Application | Data/content addressable by URI | A file, a DB row, a webpage snapshot |
| **Prompts** | User | Reusable, parameterized prompt templates | `/summarize_report`, `/code_review` |

**Transport layers:** `stdio` (local subprocess) for local tools; `HTTP + Server-Sent Events (SSE)` for remote servers.

SDKs are available in Python, TypeScript, C#, and Java.

### MCP Adoption — One Year In (as of late 2025)

([Source: Anthropic](https://www.anthropic.com/news/model-context-protocol) | [Wikipedia](https://en.wikipedia.org/wiki/Model_Context_Protocol))

| Metric | Value |
|--------|-------|
| Monthly SDK downloads | **97 million+** |
| Community MCP servers | **5,800+** |
| MCP client implementations | **300+** |
| OpenAI adoption | March 2025 (integrated into ChatGPT desktop) |
| Google DeepMind adoption | April 2025 (Gemini models) |
| Microsoft adoption | Azure OpenAI + Microsoft 365 |
| Governance | Donated to Linux Foundation (AAIF), Dec 2025 |

Early adopters: **Block, Apollo, Zed, Replit, Codeium, Sourcegraph**, and hundreds of community contributors.
Pre-built MCP servers exist for **GitHub, Slack, PostgreSQL, Google Drive, Brave Search**, and many more.

MCP evolved from an Anthropic internal tool into common infrastructure co-governed by Anthropic, OpenAI,
Google, Microsoft, AWS, Cloudflare, and Bloomberg under the Linux Foundation.

### Why It Matters for Multi-Agent Systems

Without MCP (tightly coupled):
```python
# Each agent hardcodes the tool's import — agents depend on implementation details
from my_budget_lib import estimate_launch_budget
result = estimate_launch_budget(students=2000)
```

With MCP (loosely coupled):
```python
# Agent calls through a standard interface — doesn't know or care about implementation
result = mcp_call('estimate_launch_budget', students=2000)
```

**Benefits at scale:**
- Swap tool implementations without touching any agent code
- One place for logging, authentication, and rate limiting
- Easy to mock tools in tests (just replace the registry)
- Centralized, consistent error handling

**📝 TODO:** Implement at least one more tool (e.g., `estimate_support_load`), add it to `TOOLS`,
and strengthen `mcp_call` with type validation or fallback handling.

In [9]:
def estimate_launch_budget(students: int) -> int:
    """Estimate budget needed for launch ($/student)."""
    # TODO: Replace with your own model or fetch from DB
    if students <= 0:
        raise ValueError('students must be > 0')
    return students * 12  # $12 per student


# TODO: Add at least one more tool
def estimate_support_load(students: int) -> int:
    """Estimate number of support staff needed (1 per N students)."""
    if students <= 0:
        raise ValueError('students must be > 0')
    return max(1, students // 80)  # 1 staff per 80 students


TOOLS = {
    'estimate_launch_budget': estimate_launch_budget,
    'estimate_support_load': estimate_support_load,
}


def mcp_call(tool_name: str, **kwargs):
    """Standard MCP interface for calling tools.
    
    TODO: Add better error handling:
    - Validate argument types before calling
    - Provide friendly error messages
    - Consider retries or fallbacks
    """
    tool = TOOLS.get(tool_name)
    if tool is None:
        raise ValueError(f'Unknown tool: {tool_name}')
    try:
        return tool(**kwargs)
    except TypeError as e:
        raise ValueError(f'Invalid arguments for {tool_name}. Expected kwargs but got {kwargs}: {e}')


print('✓ MCP-style tool adapter ready')

✓ MCP-style tool adapter ready


In [ ]:
# ─── Test your MCP tool interface ───────────────────────────────────────────
# Run this cell to verify your tools work correctly before building the graph.
# Both the happy-path and the error cases should behave as expected.

print("=== MCP Tool Tests ===\n")

# Happy path: valid inputs
budget = mcp_call('estimate_launch_budget', students=500)
support = mcp_call('estimate_support_load', students=500)
print(f"estimate_launch_budget(500 students)  →  ${budget:,}")
print(f"estimate_support_load(500 students)   →  {support} staff member(s)")

# Larger cohort
budget_2k = mcp_call('estimate_launch_budget', students=2000)
print(f"estimate_launch_budget(2000 students) →  ${budget_2k:,}")

print("\n--- Error Handling Tests ---")

# Unknown tool name
try:
    mcp_call('nonexistent_tool', foo='bar')
except ValueError as e:
    print(f"Unknown tool error caught ✓  →  {e}")

# Invalid argument (negative students)
try:
    mcp_call('estimate_launch_budget', students=-5)
except (ValueError, Exception) as e:
    print(f"Invalid argument error caught ✓ →  {e}")

print("\n✓ MCP interface handles both success and error cases correctly")
print("  Your agents will call tools through this same interface — errors surface cleanly.")

## Part B: Agent Nodes with Control Logic

### State-Based Routing Concept

Instead of always following A → B → C, we make decisions based on state:

```
  coordinator (create draft, decide: do we need tools?)
       ↓
  tool_user (try to call tools; on error, bail out)
       ↙         ↖
      ✓           ✗ (error)
   reviewer       failover (return error output)
       ↓              ↓
  quality OK?       END
   ↙        ↖
  ✓          ✗ (low quality)
finalize    reviser (improve draft)
  ↓            ↓
 END     reviewer (re-grade)
           (loop until quality OK or max retries hit)
```

**Key decisions:**
- `tool_user → failover OR reviewer?` (Did tool call succeed?)
- `reviewer → finalize OR reviser?` (Is quality high enough?)
- `reviser → reviewer?` (Retry loop, but cap at max_retries)

**📝 TODO:** Implement coordinator, tool_user, reviewer, reviser, finalize, and failover functions. Each should update state and add a log message. Use state fields to make routing decisions.

In [10]:
def coordinator(state: ControlState) -> ControlState:
    # TODO: Improve draft generation—make it specific to state['task']
    state['draft'] = f"Draft plan: {state['task']}"
    state['needs_tool'] = 'budget' in state['task'].lower() or 'cost' in state['task'].lower()
    state['log'].append('coordinator: created initial draft')
    return state


def tool_user(state: ControlState) -> ControlState:
    """Try to call tools. If error, set state['error'] so routing can handle it."""
    try:
        if state['needs_tool']:
            # TODO: Call other tools as needed (e.g., estimate_support_load)
            budget = mcp_call('estimate_launch_budget', students=2000)
            support = mcp_call('estimate_support_load', students=2000)
            state['draft'] += f" | Budget: ${budget} | Support staff: {support}"
            state['log'].append('tool_user: tool calls succeeded')
        else:
            state['log'].append('tool_user: tools not needed for this task')
    except Exception as exc:
        state['error'] = str(exc)
        state['log'].append(f"tool_user error: {exc}")
    return state


def reviewer(state: ControlState) -> ControlState:
    """Grade the draft quality on a 0–1 scale."""
    # TODO: Improve scoring—make it task-specific
    quality = 0.50
    if state['protocol'] == 'two_way':
        quality += 0.15
    elif state['protocol'] == 'brainstorm':
        quality += 0.25
    if 'Budget:' in state['draft']:
        quality += 0.10
    if 'Support staff:' in state['draft']:
        quality += 0.08
    state['quality_score'] = min(quality, 0.99)
    state['feedback'] = 'Pass' if state['quality_score'] >= 0.75 else 'Needs revision'
    state['log'].append(f"reviewer: quality={state['quality_score']:.2f}, feedback={state['feedback']}")
    return state


def reviser(state: ControlState) -> ControlState:
    """Revise draft to improve quality."""
    state['retries'] += 1
    state['draft'] += f" [Revision {state['retries']}: Added KPI targets and owner assignments]"
    state['log'].append(f"reviser: revision #{state['retries']} applied")
    return state


def finalize(state: ControlState) -> ControlState:
    """Output is ready; mark as final."""
    state['final_output'] = state['draft']
    state['log'].append('finalize: workflow complete')
    return state


def failover(state: ControlState) -> ControlState:
    """Tool or workflow error; return fallback."""
    state['final_output'] = f"[ERROR] {state['error']}"
    state['log'].append('failover: error recovery applied')
    return state


print('✓ All node functions defined')

✓ All node functions defined


## Part C: Conditional Routing and Iteration Loops

### Conditional Edges in LangGraph

Unlike `graph.add_edge(A, B)` (A always goes to B), **conditional edges** evaluate state and choose the next node:

```python
graph.add_conditional_edges(
    'tool_user',  # From this node
    route_after_tool,  # Call this function to decide
    {  # Map of return values → next nodes
        'reviewer': 'reviewer',
        'failover': 'failover',
    }
)
```

The `route_after_tool` function reads state and returns the next node name.

**📝 TODO:** Implement routing functions that decide which node to visit next based on error and quality_score. Then build the graph with conditional edges.

In [11]:
def route_after_tool(state: ControlState) -> str:
    """Decide: if error, go to failover; otherwise reviewer."""
    if state['error']:
        return 'failover'
    return 'reviewer'


def route_after_review(state: ControlState) -> str:
    """Decide: if quality OK, finalize; if retries left, revise; otherwise finalize anyway."""
    # TODO: Experiment with different thresholds (0.7, 0.75, 0.8, etc.)
    if state['quality_score'] >= 0.75:
        return 'finalize'
    if state['retries'] >= state['max_retries']:
        return 'finalize'  # Give up after max retries
    return 'reviser'  # Try again


def build_graph():
    graph = StateGraph(ControlState)
    
    # Add all nodes
    graph.add_node('coordinator', coordinator)
    graph.add_node('tool_user', tool_user)
    graph.add_node('reviewer', reviewer)
    graph.add_node('reviser', reviser)
    graph.add_node('finalize', finalize)
    graph.add_node('failover', failover)
    
    # Add edges: start → coordinator
    graph.add_edge(START, 'coordinator')
    graph.add_edge('coordinator', 'tool_user')
    
    # Conditional edge: tool_user → reviemr OR failover
    graph.add_conditional_edges(
        'tool_user',
        route_after_tool,
        {'reviewer': 'reviewer', 'failover': 'failover'}
    )
    
    # Conditional edge: reviewer → finalize OR reviser
    graph.add_conditional_edges(
        'reviewer',
        route_after_review,
        {'finalize': 'finalize', 'reviser': 'reviser'}
    )
    
    # Loop: reviser → reviewer (for retry)
    graph.add_edge('reviser', 'reviewer')
    
    # Terminal edges
    graph.add_edge('finalize', END)
    graph.add_edge('failover', END)
    
    return graph.compile()


compiled_graph = build_graph()
print('✓ Conditional graph compiled')

✓ Conditional graph compiled


In [ ]:
# Visualize the conditional graph structure
try:
    from IPython.display import Image, display
    img = compiled_graph.get_graph().draw_mermaid_png()
    display(Image(img))
    print("✓ Graph rendered above")
except Exception:
    print("""
    Conditional Workflow (ASCII diagram)
    ─────────────────────────────────────────────────────────────

              ┌─────────────┐
              │    START    │
              └──────┬──────┘
                     │
                     ▼
            ┌─────────────────┐
            │   coordinator   │  creates draft; sets needs_tool flag
            └────────┬────────┘
                     │
                     ▼
            ┌─────────────────┐
            │    tool_user    │  calls MCP tools (or skips if not needed)
            └────────┬────────┘
           ✓ OK      │        ✗ Error
         ┌───────────┴───────────┐
         ▼                       ▼
  ┌─────────────┐       ┌──────────────────┐
  │   reviewer  │       │     failover     │ ──→ END
  └──────┬──────┘       └──────────────────┘
         │
   quality ≥ threshold?
         │
  ┌──────┴────────────┐
  ▼ (pass)            ▼ (fail, retries left)
┌──────────┐     ┌──────────┐
│ finalize │     │ reviser  │ ──→ reviewer  (retry loop)
└────┬─────┘     └──────────┘
     │
     ▼
    END

    Key routing decisions:
      route_after_tool()  → failover if error, else reviewer
      route_after_review() → finalize if quality OK or retries exhausted, else reviser
    """)

## Part D: Benchmark Coordination Configurations

### Why Benchmark?

Different strategies have trade-offs:

| Strategy | Quality | Reliability | Speed | Cost |
|----------|---------|-------------|-------|------|
| One-way | Medium | Lower | Fast | Low |
| Two-way | Higher | Medium | Slow | Medium |
| Brainstorm | Highest | Lower | Slowest | High |
| +Iteration | +10% | +20% | −50% | +30% |

**Benchmarking answers:** *Which strategy is right for **my constraints**?*

### What the Research Says: Protocol Choice Has Measurable Impact

**"Which LLM Multi-Agent Protocol to Choose?" (2024)** ([arXiv 2510.17149](https://arxiv.org/pdf/2510.17149))

A dedicated study on multi-agent protocol selection found:

| Metric | Finding |
|--------|---------|
| Task completion time delta | Protocol choice alone impacts this by **up to 36%** |
| Communication overhead | Up to **3.5 seconds** per agent interaction depending on protocol |
| Hybrid protocol advantage | Hybrid deployments (mixing protocols) outperformed homogeneous by **~6.6%** |
| Routing improvement | A "ProtocolRouter" that dynamically selects protocols improved performance by **up to 18%** |

> **Implication:** there is no universally best protocol. The right protocol depends on task type,
> latency constraints, and agent team composition — which is why you benchmark, not guess.

### LangChain Architecture Benchmark (2024)
([Source: blog.langchain.com](https://blog.langchain.com/benchmarking-multi-agent-architectures/))

Using the τ-bench dataset with distractor domains:
- **Swarm** slightly outperformed **Supervisor** across all distractor counts
- **Single Agent** degraded sharply beyond 2 distractor domains
- Supervisor improved by **nearly 50%** after three implementation optimizations
- Both Supervisor and Swarm had **flat token usage** as complexity scaled — Single Agent did not

### What Real-World Systems Benchmark

Your proxy metrics below map directly to what production teams track:

| Your proxy metric | What it approximates | Real-world equivalent |
|-------------------|---------------------|-----------------------|
| `quality_score` | Output usefulness / correctness | Human eval score, automated rubric, task success rate |
| `retries` | Revision cycles required | LLM API calls, latency added by iteration |
| `log_steps` | Total coordination actions | Token usage proxy (each message ≈ context growth) |
| `efficiency` | Quality per unit of effort | Cost-per-quality-point; ROI of coordination complexity |
| `status` | Did the workflow complete? | Reliability / error rate in production |

### Benchmark Systems in the Literature

- **AgentBench** (Liu et al., 2023, [arXiv 2308.03688](https://arxiv.org/abs/2308.03688)): First systematic benchmark for LLM agents
  across 8 task types (OS, databases, web, card games). Showed GPT-4 significantly outperforms
  open-source models on agentic tasks, especially those requiring multi-step error recovery.

- **MultiAgentBench** (2025, [arXiv 2503.01935](https://arxiv.org/html/2503.01935v1)): Specifically evaluates collaboration and
  competition dynamics in LLM multi-agent systems — the first benchmark designed around inter-agent
  interaction quality, not just task outcomes.

- **MagenticOne** (Microsoft, 2024, [arXiv 2411.04468](https://arxiv.org/abs/2411.04468)): A generalist multi-agent system using an
  Orchestrator + 4 specialist agents. Evaluated on GAIA, AssistantBench, and WebArena. The Orchestrator
  (supervisor) pattern outperformed flat peer-to-peer coordination on structured tasks.

**📝 TODO:** Run each protocol on 2–3 task variants (with/without budget needs). Build a DataFrame
and compute mean metrics by protocol. Which is most efficient? Most reliable?

In [12]:
def run_config(protocol: str, task_text: str, max_retries: int = 2):
    """Run a single configuration and return results."""
    state: ControlState = {
        'task': task_text,
        'protocol': protocol,
        'draft': '',
        'feedback': '',
        'final_output': '',
        'quality_score': 0.0,
        'retries': 0,
        'max_retries': max_retries,
        'needs_tool': False,
        'error': '',
        'log': []
    }
    result = compiled_graph.invoke(state)
    complexity = result['retries'] + len(result['log'])
    return {
        'protocol': protocol,
        'task': task_text,
        'quality_score': result['quality_score'],
        'retries': result['retries'],
        'log_steps': len(result['log']),
        'complexity': complexity,
        'efficiency': result['quality_score'] / max(complexity, 1),
        'status': 'failed' if result['error'] else 'ok'
    }


# Define test tasks
tasks = [
    'Create a launch plan with timeline and owner assignments.',
    'Create a launch plan and estimate required budget for 2000 students.',
    'Create a launch plan with staffing needs and resource constraints.',
]

protocols = ['one_way', 'two_way', 'brainstorm']

# Run all configurations
rows = []
for task_text in tasks:
    for protocol in protocols:
        row = run_config(protocol, task_text, max_retries=2)
        rows.append(row)

df = pd.DataFrame(rows)
print('\n=== Full Results ===')
display(df)

# Aggregate by protocol
summary = df.groupby('protocol')[['quality_score', 'complexity', 'efficiency', 'retries']].mean().round(3)
print('\n=== Aggregated by Protocol ===')
display(summary)

print('\n=== Interpretation ===')
print(f"Best quality: {summary['quality_score'].idxmax()}")
print(f"Lowest complexity: {summary['complexity'].idxmin()}")
print(f"Highest efficiency: {summary['efficiency'].idxmax()}")


=== Full Results ===


,protocol,task,quality_score,retries,log_steps,complexity,efficiency,status
0,one_way,Create a launch plan with timeline and owner a...,0.50,2,8,10,0.0500,ok
1,two_way,Create a launch plan with timeline and owner a...,0.65,2,8,10,0.0650,ok
2,brainstorm,Create a launch plan with timeline and owner a...,0.75,0,4,4,0.1875,ok
3,one_way,Create a launch plan and estimate required bud...,0.68,2,8,10,0.0680,ok
4,two_way,Create a launch plan and estimate required bud...,0.83,0,4,4,0.2075,ok
5,brainstorm,Create a launch plan and estimate required bud...,0.93,0,4,4,0.2325,ok
6,one_way,Create a launch plan with staffing needs and r...,0.50,2,8,10,0.0500,ok
7,two_way,Create a launch plan with staffing needs and r...,0.65,2,8,10,0.0650,ok
8,brainstorm,Create a launch plan with staffing needs and r...,0.75,0,4,4,0.1875,ok



=== Aggregated by Protocol ===


,quality_score,complexity,efficiency,retries
protocol,,,,
brainstorm,0.81,4.0,0.202,0.000
one_way,0.56,10.0,0.056,2.000
two_way,0.71,8.0,0.112,1.333



=== Interpretation ===
Best quality: brainstorm
Lowest complexity: brainstorm
Highest efficiency: brainstorm


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ─── Benchmark Visualization ────────────────────────────────────────────────
# Build a per-protocol summary from the benchmark DataFrame

protocols_ordered = ['one_way', 'two_way', 'brainstorm']
summary_plot = df.groupby('protocol')[['quality_score', 'complexity', 'efficiency', 'retries']].mean()
summary_plot = summary_plot.reindex(protocols_ordered)

colors = ['#4C9BE8', '#F5A623', '#7ED321']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Module 5 — Part 2 Multi-Agent Coordination Benchmark', fontsize=13, fontweight='bold', y=1.02)

# --- Quality Score ---
ax1 = axes[0]
bars1 = ax1.bar(protocols_ordered, summary_plot['quality_score'], color=colors, width=0.5)
ax1.set_title('Average Quality Score', fontweight='bold')
ax1.set_ylabel('Quality (0 – 1)')
ax1.set_ylim(0, 1.1)
ax1.axhline(y=0.75, color='red', linestyle='--', alpha=0.6, label='Pass threshold (0.75)')
ax1.legend(fontsize=8)
for bar, val in zip(bars1, summary_plot['quality_score']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

# --- Complexity Proxy ---
ax2 = axes[1]
bars2 = ax2.bar(protocols_ordered, summary_plot['complexity'], color=colors, width=0.5)
ax2.set_title('Average Complexity\n(retries + log steps)', fontweight='bold')
ax2.set_ylabel('Complexity units')
ax2.set_ylim(0, summary_plot['complexity'].max() * 1.15 + 1)
for bar, val in zip(bars2, summary_plot['complexity']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

# --- Efficiency (quality per complexity unit) ---
ax3 = axes[2]
bars3 = ax3.bar(protocols_ordered, summary_plot['efficiency'], color=colors, width=0.5)
ax3.set_title('Average Efficiency\n(quality ÷ complexity)', fontweight='bold')
ax3.set_ylabel('Efficiency score')
ax3.set_ylim(0, summary_plot['efficiency'].max() * 1.15 + 0.005)
for bar, val in zip(bars3, summary_plot['efficiency']):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
             f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print(f"  Best quality  : {summary_plot['quality_score'].idxmax()}")
print(f"  Lowest complexity: {summary_plot['complexity'].idxmin()}")
print(f"  Highest efficiency: {summary_plot['efficiency'].idxmax()}")
print("\n🔍 Pause & Discuss:")
print("   1. Does higher quality always justify higher complexity in your use case?")
print("   2. How does the revision loop change results for each protocol?")
print("   3. When would you set max_retries to 0?  When would you raise it to 5?")

### Try Your Own Task (Part 2)

Once you are comfortable with the benchmark on the provided tasks, adapt it:

1. Add one or two of your own tasks to the `tasks` list in Part 2.
2. Re-run the benchmark loop that constructs `rows` and the `df` DataFrame.
3. Inspect how quality, retries, complexity, and efficiency change for your tasks across protocols.
4. Decide which protocol and `max_retries` setting you would deploy, and what failover behavior you would require.

### Quick Start Demo (Part 2)

Once you have run the Part 2 setup cells (imports, `ControlState`, MCP tools, nodes, routing, graph build, and benchmark loop),
you can use this cell to quickly run one configuration and inspect its metrics.

- Start with the provided protocol and task, then modify them to see different behaviors.

In [ ]:
# Quick start: run Part 2 once for a single protocol and task
example_protocol = 'brainstorm'
example_task = 'Create a launch plan and include a budget estimate for 2000 students.'

row, result = run_config(example_protocol, example_task, max_retries=2)

print(f"Protocol: {row['protocol']}")
print(f"Task    : {row['task']}\n")
print("Final output (truncated to first 800 chars):\n")
print(result['final_output'][:800])
print("\nSummary metrics:")
print(f"  quality_score   = {row['quality_score']:.2f}")
print(f"  retries         = {row['retries']}")
print(f"  log_steps       = {row['log_steps']}")
print(f"  complexity      = {row['complexity']}")
print(f"  efficiency      = {row['efficiency']}")

## Reflection & Production Decisions

Use your benchmark results to answer:

1. **Trade-off Winner:** Which protocol had the best quality-to-complexity trade-off?
   - Plot or rank by efficiency score
   - When would you choose each?

2. **Iteration ROI:** How much did the revision loop (coordinator → tool → reviewer → reviser) improve quality?
   - Compare: quality without retries vs. with retries
   - Did max_retries=2 feel too low, too high, or about right?
   - Would adaptive retry thresholds help?

3. **Error Handling Strategy:** In what scenarios should failover be strict vs. permissive?
   - **Strict:** Fast failure, simple recovery, but may lose quality (risky)
   - **Permissive:** Retry/fallback, attempt recovery, but slower (safer)
   - Which fits your use case?

4. **MCP Value:** How does standardized tool access influence your architecture?
   - Would you add more tools if interface were standard?
   - How would centralized logging help?
   - Could you test agents more easily?

5. **Scaling to Production:** If deploying to a real team:
   - **Which protocol to start with?** (Likely one-way for speed, two-way for quality)
   - **When to add iteration?** (If quality variance is high, always)
   - **How to monitor?** (Log all tool calls, quality scores, and error rates)
   - **How to adjust?** (A/B test different thresholds, collect user feedback)

6. **Context Window Trade-off:** How would token limits change your choices?
   - Two-way communication grows message_log faster
   - Brainstorm explodes token usage
   - At what team size must you switch to one-way or hierarchical routing?

---
## Summary & Key Concepts

**What you built in Part 2:**

```
Simple (Part 1)                          Adaptive (Part 2)
─────────────────────     →     ──────────────────────────────────
START → A → B → C → END          START → coordinator → tool_user
                                              ↙ (error)  ↘ (OK)
                                          failover     reviewer
                                                        ↙    ↘
                                                  finalize  reviser ↻
```

**Pattern cheatsheet:**

| Pattern | LangGraph construct | When to use |
|---------|-------------------|-------------|
| Linear chain | `add_edge(A, B)` | Simple, fully trusted pipelines |
| Conditional fork | `add_conditional_edges(node, fn, map)` | Error handling, quality gating |
| Iteration loop | `reviser → reviewer → reviser` | Quality-driven improvement with retry cap |
| Tool standardization | `mcp_call(name, **kwargs)` | Multiple tools shared across agents |

**MCP in multi-agent systems:**

The Model Context Protocol (MCP) standardizes how agents register and invoke tools so that
any agent can call any tool through the same interface — without knowing the implementation.
At scale, a 10-agent workflow calling 20 different tools needs a standard interface to stay
maintainable, testable, and monitorable.

**Coordination vs. complexity trade-off:**

> Adding adaptive control (iteration, error handling) improves robustness but raises complexity.
> Your benchmark showed this empirically.
> **Rule of thumb:** start with a simple linear graph, add conditional edges only when you can
> measure the quality benefit that justifies the added complexity.

**Before you move on from Part 2, you should be able to:**
- Explain how the routing functions decide between `reviewer`, `reviser`, `finalize`, and `failover`.
- Read the benchmark DataFrame and plots to compare protocols on your own tasks.
- State when you would change `max_retries` or adjust the quality threshold in a production workflow.

---

**You have now completed both Module 5 labs.**
The concepts from these labs — agent design, coordination protocols, conditional control flow,
MCP tool access, and benchmarking — form the foundation for building reliable multi-agent systems
in production.

---
## Further Reading & Resources

Use these to go deeper on any topic from this module:

### Multi-Agent Architecture & Coordination
- [LangGraph Multi-Agent Concepts](https://langchain-ai.github.io/langgraph/concepts/multi_agent/) — Official docs on Supervisor, Swarm, and Handoff patterns
- [Benchmarking Multi-Agent Architectures](https://blog.langchain.com/benchmarking-multi-agent-architectures/) — LangChain's empirical comparison of Swarm vs. Supervisor vs. Single Agent
- [Agent Orchestration Patterns: Swarm vs Mesh vs Hierarchical vs Pipeline](https://dev.to/jose_gurusup_dev/agent-orchestration-patterns-swarm-vs-mesh-vs-hierarchical-vs-pipeline-b40) — Visual breakdown of production topology patterns

### Research Papers
- [More Agents Is All You Need](https://arxiv.org/abs/2402.05120) (Li et al., 2024) — Majority-vote ensemble with smaller models beats larger single models
- [Which LLM Multi-Agent Protocol to Choose?](https://arxiv.org/pdf/2510.17149) (2024) — Protocol selection impacts task completion time by up to 36%
- [A Communication-Centric Survey of LLM-Based Multi-Agent Systems](https://arxiv.org/pdf/2502.14321) (2025) — Comprehensive framework for analyzing agent communication
- [MultiAgentBench](https://arxiv.org/html/2503.01935v1) (2025) — Benchmark for evaluating LLM agent collaboration and competition
- [AgentBench](https://arxiv.org/abs/2308.03688) (Liu et al., 2023) — Foundational benchmark across 8 agentic task types

### Model Context Protocol (MCP)
- [Anthropic MCP Announcement](https://www.anthropic.com/news/model-context-protocol) — Original announcement with architecture and early adopters
- [MCP Official Documentation](https://modelcontextprotocol.io/specification/2025-11-25) — Current specification
- [MCP GitHub Organization](https://github.com/modelcontextprotocol) — SDKs, reference servers, and community contributions
- [One Year of MCP: November 2025 Review](http://blog.modelcontextprotocol.io/posts/2025-11-25-first-mcp-anniversary/) — Ecosystem stats: 97M+ monthly downloads, 5,800+ servers

### Industry Reports
- [LangChain State of AI Agents (2024)](https://www.langchain.com/stateofaiagents) — 1,300+ professional survey: adoption rates, use cases, workflow complexity trends

---
## Troubleshooting

If you hit errors while working through this notebook, try these steps first:

- **`ModuleNotFoundError` for `langgraph` or `pandas`**
  - Re-run the install cells near the start of each part (the ones beginning with `!pip install`).
  - If you just installed packages in a fresh environment, restart the kernel and then *Run All*.

- **Plots not appearing**
  - Confirm the plotting cells at the end of each part run without errors.
  - In some environments, running `%matplotlib inline` once at the top of the notebook can help.

- **NameError for functions or classes (e.g., `run_protocol`, `ControlState`, `compiled_graph`)**
  - Make sure all earlier setup cells in that part have been executed in order.
  - Use *Kernel → Restart & Run All* to reset execution order if things feel out of sync.

- **Empty benchmark results or `df` is undefined in Part 2**
  - Re-run the cells that build the `rows` list and construct the `df` DataFrame before the visualization.

If an issue persists, try printing intermediate values (like a single row from `rows` or part of the state dict)
to see exactly where execution diverges from what you expect.